In [0]:
# Leemos la tabla de clientes de bronze y la guardamos en silver
clientes_bronze_df = spark.read.table("retail.01_bronze.clientes_df")

clientes_bronze_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.02_silver.clientes")


In [0]:
# Cargar la tabla a un dataframe y lo pasas a pandas
df_pd = spark.table("retail.02_silver.clientes").toPandas()

# Muestro el conteo de datos no nulos por campo
display(df_pd.count())

ID_Cliente      992
Edad            992
Sexo            992
EstCivil        987
Segmento_RFM    992
dtype: int64

In [0]:
#--------------------------Limpieza tabla Clientes----------------------------------------

# Paso 1: Cargar la tabla clientes
clientes_df = spark.table("retail.02_silver.clientes")

# Paso 2: Eliminar duplicados
clientes_df = clientes_df.dropDuplicates(["ID_Cliente"])

# Paso 3: Eliminar filas con valores nulos en columnas clave
clientes_df = clientes_df.dropna(subset=["ID_Cliente", "Edad", "Sexo", "EstCivil", "Segmento_RFM"])

# Paso 4: Filtrar edades fuera de rango (por ejemplo, solo edades entre 18 y 100)
clientes_df = clientes_df.filter((clientes_df.Edad >= 18) & (clientes_df.Edad <= 100))

# Paso 5: Normalizar valores de Sexo (por ejemplo, solo 'M' o 'F')
clientes_df = clientes_df.filter(clientes_df.Sexo.isin("Masculino", "Femenino"))

# Paso 6: Revisar valores válidos en EstCivil
clientes_df = clientes_df.filter(clientes_df.EstCivil.isin("SOLTERO", "CASADO", "DIVORCIADO", "VIUDO"))


In [0]:
clientes_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.02_silver.clientes")

In [0]:
# Cargas la tabla a un dataframe y la pasas a pandas
a_pd = spark.table("retail.02_silver.clientes").toPandas()

# Muestras el conteo de datos no nulos por campo
display(a_pd.count())

ID_Cliente      987
Edad            987
Sexo            987
EstCivil        987
Segmento_RFM    987
dtype: int64

In [0]:
%sql
select * from retail.`02_silver`.clientes
limit(5)

ID_Cliente,Edad,Sexo,EstCivil,Segmento_RFM
63529974,53,Masculino,CASADO,Ocasional
49173228,68,Masculino,CASADO,Ocasional
70795246,50,Femenino,SOLTERO,Ocasional
58592656,51,Femenino,CASADO,Habitual
69437503,45,Masculino,CASADO,Ocasional


In [0]:
# Leemos la tabla de clientes de bronze y la guardamos en silver
ventas_bronze_df = spark.read.table("retail.01_bronze.ventas_df")
ventas_bronze_df = ventas_bronze_df.drop("Mes","Marca","Dia")

ventas_bronze_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.02_silver.ventas")
    

In [0]:
# Cargas la tabla a un dataframe y la pasas a pandas
df_pd = spark.table("retail.02_silver.ventas").toPandas()

# Muestras el conteo de datos no nulos por campo
display(df_pd.count())

CodTicket      446400
ID_Cliente     446400
CodMaterial    446400
Cod_N1         446400
Desc_N1        446400
N2             446400
N3             446400
VTABS          446400
Margen         446400
VTAQ           446400
IGV            446400
dtype: int64

In [0]:
# Verificar cuántos valores negativos hay en el campo VTABS
ventas_df = spark.table("retail.02_silver.ventas")
negativos_df = ventas_df.filter(ventas_df.VTABS < 0)
display(negativos_df.limit(5))

CodTicket,ID_Cliente,CodMaterial,Cod_N1,Desc_N1,N2,N3,VTABS,Margen,VTAQ,IGV
S23-23/04/2016-0005-0074,61668147,390716,1010104,RES NACIONAL CORTE PRIMARIO BISTECKS,CARNES ROJAS RES,FOOD PERECIBLES,-10.0799974,-0.690420926,-0.548719993,1.18006034
S23-15/06/2016-0002-0100,27436456,373511,1010108,RES NACIONAL CORTE SECUNDARIO GUISOS,CARNES ROJAS RES,FOOD PERECIBLES,-17.78666242,-0.370588235,-1.083,1.259326704
S23-13/06/2017-0003-0160,75656690,363422,5010101,TUBERCULOS Y RAICES,VERDURAS,FOOD PERECIBLES,-4.039998952,-0.010311111,-1.256280007,1.047154581
S23-04/01/2017-0004-0244,59745667,227888,2010201,POLLO CORTES PECHUGA,POLLOS,FOOD PERECIBLES,-20.18666208,-3.223255713,-1.212959962,1.18008961
S23-26/12/2016-0003-0226,63530014,227888,2010201,POLLO CORTES PECHUGA,POLLOS,FOOD PERECIBLES,-34.82665908,-4.529523958,-2.093800069,1.254814465


In [0]:
from pyspark.sql import functions as F

null_counts = ventas_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(f"{c}_nulls") 
    for c in ["CodTicket", "ID_Cliente", "CodMaterial", "VTABS"]
])
display(null_counts)

CodTicket_nulls,ID_Cliente_nulls,CodMaterial_nulls,VTABS_nulls
0,0,0,0


In [0]:
# ------------ Limpieza tabla Ventas ---------------------

# 4. Renombrar campos para la capa Silver (estandarizando nombres de categorías)
ventas_df = ventas_df \
    .withColumnRenamed("Cod_N1", "SKU") \
    .withColumnRenamed("Desc_N1", "Producto") \
    .withColumnRenamed("N2", "Descripción") \
    .withColumnRenamed("N3", "Tipo")

# 2. Convertir valores negativos a positivos
ventas_df = ventas_df.withColumn("VTABS", F.abs(F.col("VTABS"))) \
    .withColumn("Margen", F.abs(F.col("Margen"))) \
    .withColumn("VTAQ", F.abs(F.col("VTAQ")))

# 7. Guardar el resultado limpio en la capa Silver
ventas_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail.02_silver.ventas")

In [0]:
# Cargas la tabla a un dataframe y la pasas a pandas
df_pd = spark.table("retail.02_silver.ventas").toPandas()

# Muestras el conteo de datos no nulos por campo
display(df_pd.count())

CodTicket      446400
ID_Cliente     446400
CodMaterial    446400
SKU            446400
Producto       446400
Descripción    446400
Tipo           446400
VTABS          446400
Margen         446400
VTAQ           446400
IGV            446400
dtype: int64

In [0]:
%sql
select * from retail.`02_silver`.ventas
limit(5)

CodTicket,ID_Cliente,CodMaterial,SKU,Producto,Descripción,Tipo,VTABS,Margen,VTAQ,IGV
S23-15/12/2016-0003-0163,55606046,390617,1010104,RES NACIONAL CORTE PRIMARIO BISTECKS,CARNES ROJAS RES,FOOD PERECIBLES,13.69332973,0.859806877,0.693119985,1.183367779
S23-04/12/2016-0003-0085,50592449,390617,1010104,RES NACIONAL CORTE PRIMARIO BISTECKS,CARNES ROJAS RES,FOOD PERECIBLES,18.91999471,0.025082124,0.851959962,1.182688086
S23-26/12/2016-0002-0195,61668147,390617,1010104,RES NACIONAL CORTE PRIMARIO BISTECKS,CARNES ROJAS RES,FOOD PERECIBLES,11.18666425,0.050589092,0.534280007,1.123569539
S23-08/10/2016-0004-0129,75331922,109178,1010107,RES NACIONAL CORTE SECUNDARIO MOLIDAS,CARNES ROJAS RES,FOOD PERECIBLES,5.333332,0.036437958,0.519840021,1.194107858
S23-08/10/2016-0005-0082,23450618,109178,1010107,RES NACIONAL CORTE SECUNDARIO MOLIDAS,CARNES ROJAS RES,FOOD PERECIBLES,5.879998327,0.040486619,0.577600009,1.184853511
